# Curve fitting and calibration with `scipy.optimize`

Fitting a model with unknown parameters to data: `curve_fit`, `least_squares`, `minimize`.
Every idea is shown first on six numbers you can check by hand, then on the hourly power data.

**What's in here**
- a model function, its residuals, and the sum of squared errors (SSE)
- `curve_fit`: `popt`, `pcov`, standard errors
- a bad starting point: the optimiser "converges" to the wrong answer
- `least_squares`: the same fit with the residual vector and Jacobian exposed
- bounds, and what a parameter sitting on a bound means
- standard errors by hand, and why they are too small when residuals are autocorrelated
- robust losses when there is a spike
- variable projection: linear parameters with `lstsq`, non-linear ones with `minimize`
- identifiability: profile the SSE over one parameter
- `brentq`: invert a curve
- scaling: why optimisers fail when numbers are very large or very small
- maximum likelihood with `minimize`
- the same fit on real data: 2022 in-sample, 2023 out-of-sample

In [1]:
import numpy as np
import pandas as pd
from scipy import optimize

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

## 1. A model function and its residuals

Demand rises when it is cold. A "heating degree" model:

`y = a + b · max(T_h − T, 0)`

Three unknowns: level `a`, slope `b`, breakpoint `T_h`. Six toy points generated from
`a = 5, b = 2, T_h = 15` with no noise.

In [2]:
x = np.array([5.0, 10.0, 15.0, 20.0, 25.0, 30.0])      # temperature
y = 2 * np.maximum(15 - x, 0) + 5                      # true a=5, b=2, T_h=15
print("x:", x)
print("y:", y)

x: [ 5. 10. 15. 20. 25. 30.]
y: [25. 15.  5.  5.  5.  5.]


At 5 °C: 15 − 5 = 10 degrees of heating, times 2, plus 5 → 25. At 15 °C and above the
`max(..., 0)` is 0, so y = 5.

The model as a Python function. `curve_fit` needs the independent variable first, then
each parameter as its own argument.

In [3]:
def model(T, a, b, Th):
    return a + b * np.maximum(Th - T, 0)

print(model(x, a=5, b=2, Th=15))        # the true parameters reproduce y exactly

[25. 15.  5.  5.  5.  5.]


A starting guess `p0` will not reproduce y. The residuals are `model − y`; the SSE is the sum
of their squares. Fitting means finding the parameters with the smallest SSE.

In [4]:
p0 = [0.0, 1.0, 10.0]
pred0 = model(x, *p0)
resid0 = pred0 - y
print("model at p0:", pred0)
print("residuals  :", resid0)
print("SSE        :", (resid0 ** 2).sum())

model at p0: [5. 0. 0. 0. 0. 0.]
residuals  : [-20. -15.  -5.  -5.  -5.  -5.]
SSE        : 725.0


## 2. `curve_fit`

`curve_fit(model, x, y, p0)` returns `popt` (best parameters) and `pcov` (their covariance
matrix). Always pass `p0`: the default is all ones, which is meaningless for a breakpoint.

In [5]:
popt, pcov = optimize.curve_fit(model, x, y, p0=p0)
print("popt:", popt)
print("SSE at popt:", ((model(x, *popt) - y) ** 2).sum())

popt: [ 5.  2. 15.]
SSE at popt: 7.917556528962303e-19


`popt` is `[5, 2, 15]`, the values we generated the data with, and the SSE is (numerically) zero.

The standard errors are the square roots of the diagonal of `pcov`. With a perfect fit they are
zero, so add a little noise to see something realistic.

In [6]:
noise = np.array([0.3, -0.2, 0.1, 0.4, -0.3, 0.2])
y_noisy = y + noise
popt, pcov = optimize.curve_fit(model, x, y_noisy, p0=p0)
se = np.sqrt(np.diag(pcov))
print("y_noisy:", y_noisy)
print("popt   :", popt)
print("pcov   :")
print(pcov)
print("std err:", se)

y_noisy: [25.3 14.8  5.1  5.4  4.7  5.2]
popt   : [ 5.1    2.1   14.619]
pcov   :
[[ 0.0217  0.     -0.0103]
 [ 0.      0.0069 -0.0235]
 [-0.0103 -0.0235  0.0944]]
std err: [0.1472 0.0833 0.3073]


`pcov` is 3×3 (one row and column per parameter). The diagonal holds the variances, so
`sqrt(diag)` gives one standard error per parameter, in the same order as `popt`.

## 3. A bad starting point

**Pitfall:** start with `T_h = 2`. Every temperature is above 2, so `max(T_h − T, 0)` is 0 for
all six points, the model is a flat line, and changing `b` or `T_h` a little changes nothing.
The optimiser sees zero gradient and stops immediately, reporting success.

In [7]:
p_bad = [0.0, 1.0, 2.0]
popt_bad, _ = optimize.curve_fit(model, x, y_noisy, p0=p_bad)
print("popt from bad start:", popt_bad)
print("SSE                :", ((model(x, *popt_bad) - y_noisy) ** 2).sum())
print("SSE from good start:", ((model(x, *popt) - y_noisy) ** 2).sum())

popt from bad start: [10.0833  1.      2.    ]
SSE                : 353.3883333333333
SSE from good start: 0.2600000000000001


/tmp/ipykernel_175460/3804344716.py:2: OptimizeWarning: Covariance of the parameters could not be estimated
  popt_bad, _ = optimize.curve_fit(model, x, y_noisy, p0=p_bad)


`b` and `T_h` did not move at all; only `a` was adjusted (to the mean of y). No error, just
an easy-to-miss `OptimizeWarning` about the covariance, and a much larger SSE. **Convergence is not correctness.** Compare the SSE from several
starting points, and compare the estimates against what you know about the problem.

## 4. `least_squares`: the same fit with more visible

`least_squares(fun, x0)` minimises `0.5 · Σ fun(θ)²`. You write the residual function
yourself; the result shows the residual vector (`fun`), the cost, the status and the Jacobian.
`curve_fit` is a wrapper around this.

In [8]:
def residuals(theta, T, y_obs):
    a, b, Th = theta
    return model(T, a, b, Th) - y_obs

res = optimize.least_squares(residuals, x0=p0, args=(x, y_noisy))
print("x (parameters):", res.x)
print("fun (residuals):", res.fun)
print("cost = 0.5*SSE :", res.cost)
print("status/message :", res.status, res.message)
print("nfev           :", res.nfev)

x (parameters): [ 5.1    2.1   14.619]
fun (residuals): [-0.  -0.   0.  -0.3  0.4 -0.1]
cost = 0.5*SSE : 0.13000000000000006
status/message : 1 `gtol` termination condition is satisfied.
nfev           : 6


`res.fun` has one entry per data point; `res.cost` is half their sum of squares.

### Bounds

`bounds=(lower, upper)` per parameter. Here the true breakpoint is 15, and we force `T_h`
to stay in [16, 30].

In [9]:
lower = [-np.inf, 0.0, 16.0]
upper = [ np.inf, 10.0, 30.0]
res_b = optimize.least_squares(residuals, x0=[0, 1, 20], bounds=(lower, upper), args=(x, y_noisy))
print("x          :", res_b.x)
print("on lower bound:", np.isclose(res_b.x, lower))
print("cost       :", res_b.cost, " (unbounded fit had", res.cost, ")")

x          : [ 4.5824  1.8337 16.    ]
on lower bound: [False False  True]
cost       : 1.8552724358974464  (unbounded fit had 0.13000000000000006 )


`T_h` sits exactly on its lower bound (16): the optimiser wanted to go lower and was stopped.
A parameter on a bound is a warning sign, always print that check.

## 5. Standard errors by hand

The usual covariance is `σ² (JᵀJ)⁻¹`, where `J` is the Jacobian (one row per data point, one
column per parameter) and `σ² = SSE / (n − p)`. This is exactly what `curve_fit` returns as `pcov`.

In [10]:
J = res.jac
print("J shape:", J.shape, "(6 points x 3 parameters)")
print(J)

J shape: (6, 3) (6 points x 3 parameters)
[[1.    9.619 2.1  ]
 [1.    4.619 2.1  ]
 [1.    0.    0.   ]
 [1.    0.    0.   ]
 [1.    0.    0.   ]
 [1.    0.    0.   ]]


In [11]:
n, p = J.shape
sse = (res.fun ** 2).sum()
sigma2 = sse / (n - p)
cov_hand = sigma2 * np.linalg.inv(J.T @ J)
print("sigma^2      :", sigma2)
print("std err hand :", np.sqrt(np.diag(cov_hand)))
print("std err pcov :", se)

sigma^2      : 0.08666666666666671
std err hand : [0.1472 0.0833 0.3073]
std err pcov : [0.1472 0.0833 0.3073]


Same numbers. The formula assumes the residuals are independent with equal variance.

**Pitfall:** with hourly data the residuals are autocorrelated (a cold hour follows a cold hour),
so there is less information than `n` suggests. A rough fix with the lag-1 autocorrelation ρ:
`n_eff = n (1 − ρ) / (1 + ρ)`, and the standard errors inflate by `sqrt(n / n_eff)`.
Ten made-up residuals with some persistence (a positive one tends to follow a positive one):

In [12]:
e = np.array([1.0, 0.5, 0.6, -0.2, -0.7, -0.3, 0.4, 0.6, -0.1, -0.5])
rho = np.corrcoef(e[1:], e[:-1])[0, 1]
n_eff = len(e) * (1 - rho) / (1 + rho)
print("e[1:] :", e[1:])
print("e[:-1]:", e[:-1])
print("rho   :", round(rho, 3))
print("n     :", len(e), " n_eff:", round(n_eff, 2))
print("inflate std errors by:", round(np.sqrt(len(e) / n_eff), 2))

e[1:] : [ 0.5  0.6 -0.2 -0.7 -0.3  0.4  0.6 -0.1 -0.5]
e[:-1]: [ 1.   0.5  0.6 -0.2 -0.7 -0.3  0.4  0.6 -0.1]
rho   : 0.493
n     : 10  n_eff: 3.4
inflate std errors by: 1.72


Positive ρ means fewer effective observations and larger true standard errors. On the real
data below ρ is about 0.86 and the inflation factor is about 3.6.

## 6. Robust losses

Least squares is dominated by outliers. Put one spike into the toy data and compare
`loss="linear"` (ordinary) with `loss="soft_l1"`, which down-weights residuals larger than
`f_scale`.

In [13]:
y_spike = y_noisy.copy()
y_spike[3] = y_spike[3] + 20        # one bad reading at x = 20
print("y_spike:", y_spike)

y_spike: [25.3 14.8  5.1 25.4  4.7  5.2]


In [14]:
res_lin = optimize.least_squares(residuals, x0=p0, args=(x, y_spike), loss="linear")
res_rob = optimize.least_squares(residuals, x0=p0, args=(x, y_spike), loss="soft_l1", f_scale=1.0)
print("linear  x:", res_lin.x)
print("soft_l1 x:", res_rob.x)
print("true     :", [5, 2, 15])

linear  x: [10.1     2.1    12.2381]
soft_l1 x: [ 5.3757  2.1    14.4878]
true     : [5, 2, 15]


In [15]:
print("residual at the spike, linear :", round(res_lin.fun[3], 2))
print("residual at the spike, soft_l1:", round(res_rob.fun[3], 2))
print("residuals elsewhere, linear   :", np.delete(res_lin.fun, 3))
print("residuals elsewhere, soft_l1  :", np.delete(res_rob.fun, 3))

residual at the spike, linear : -15.3
residual at the spike, soft_l1: -20.02
residuals elsewhere, linear   : [-0.   0.   5.   5.4  4.9]
residuals elsewhere, soft_l1  : [0.     0.     0.2757 0.6757 0.1757]


The ordinary fit moved `a` and `T_h` to chase the spike, spreading error over the good points.
The robust fit leaves a large residual on the spike (−20-ish) and fits the other five points
well. `f_scale` is in the units of the residual: set it to the size of a "normal" residual.

**Interview check:** "Why not just delete the outlier?" Deleting needs a rule for what an
outlier is, and that rule is a model too. The robust loss makes it explicit and keeps the row.

## 7. Variable projection: linear parameters linearly

Real demand also depends on the hour of day. Suppose two "hours" with their own levels
(5 and 8) plus the heating slope 2. For a **fixed** breakpoint, the model is linear in
`(level_0, level_1, b)`, so `np.linalg.lstsq` solves it exactly. Only `T_h` needs the
non-linear optimiser.

In [16]:
hour = np.array([0, 1, 0, 1, 0, 1])
level = np.array([5.0, 8.0])[hour]
y2 = level + 2 * np.maximum(15 - x, 0)
print("x    :", x)
print("hour :", hour)
print("y2   :", y2)

x    : [ 5. 10. 15. 20. 25. 30.]
hour : [0 1 0 1 0 1]
y2   : [25. 18.  5.  8.  5.  8.]


The design matrix for `T_h = 15`: one column per hour dummy, one for the heating term.

In [17]:
Th = 15.0
X = np.column_stack([hour == 0, hour == 1, np.maximum(Th - x, 0)]).astype(float)
print(X)

[[ 1.  0. 10.]
 [ 0.  1.  5.]
 [ 1.  0.  0.]
 [ 0.  1.  0.]
 [ 1.  0.  0.]
 [ 0.  1.  0.]]


In [18]:
beta, *_ = np.linalg.lstsq(X, y2, rcond=None)
sse_15 = ((X @ beta - y2) ** 2).sum()
print("beta (level0, level1, b):", beta)
print("SSE at Th=15:", round(sse_15, 6))

beta (level0, level1, b): [5. 8. 2.]
SSE at Th=15: 0.0


The linear solve recovers 5, 8 and 2 exactly. Now repeat that for several `T_h` values: the
SSE as a function of one parameter is called its **profile**.

In [19]:
rows = []
for Th in [11.0, 13.0, 15.0, 17.0, 19.0]:
    X = np.column_stack([hour == 0, hour == 1, np.maximum(Th - x, 0)]).astype(float)
    beta, *_ = np.linalg.lstsq(X, y2, rcond=None)
    sse = ((X @ beta - y2) ** 2).sum()
    rows.append([Th, round(beta[2], 3), round(sse, 3)])
pd.DataFrame(rows, columns=["Th", "b_fitted", "SSE"])

,Th,b_fitted,SSE
0,11.0,3.514,28.829
1,13.0,2.603,3.653
2,15.0,2.000,0.000
3,17.0,1.676,9.249
4,19.0,1.392,27.004


The SSE is zero at `T_h = 15` and rises on both sides: the breakpoint is well identified.
`minimize_scalar` does this search for us.

In [20]:
def sse_of_Th(Th):
    X = np.column_stack([hour == 0, hour == 1, np.maximum(Th - x, 0)]).astype(float)
    beta, *_ = np.linalg.lstsq(X, y2, rcond=None)
    return ((X @ beta - y2) ** 2).sum()

r = optimize.minimize_scalar(sse_of_Th, bounds=(8, 25), method="bounded")
print("Th:", round(r.x, 3), " SSE:", round(r.fun, 6), " success:", r.success)

Th: 15.0  SSE: 0.0  success: True


## 8. Identifiability: when the profile is flat

Now a cooling model `y = 5 + 3 · max(T − T_c, 0)` with `T_c = 22`, but only one data point is
warmer than 22. Any `T_c` between 20 and 25 fits perfectly with a different slope.

In [21]:
xc = np.array([5.0, 10.0, 15.0, 20.0, 25.0])
yc = 5 + 3 * np.maximum(xc - 22, 0)
print("xc:", xc)
print("yc:", yc)

xc: [ 5. 10. 15. 20. 25.]
yc: [ 5.  5.  5.  5. 14.]


In [22]:
rows = []
for Tc in [20.0, 21.0, 22.0, 23.0, 24.0]:
    X = np.column_stack([np.ones(5), np.maximum(xc - Tc, 0)])
    beta, *_ = np.linalg.lstsq(X, yc, rcond=None)
    sse = ((X @ beta - yc) ** 2).sum()
    rows.append([Tc, round(beta[1], 3), round(sse, 6)])
pd.DataFrame(rows, columns=["Tc", "slope_fitted", "SSE"])

,Tc,slope_fitted,SSE
0,20.0,1.80,0.0
1,21.0,2.25,0.0
2,22.0,3.00,0.0
3,23.0,4.50,0.0
4,24.0,9.00,0.0


Every row has SSE 0: the slope simply rescales to compensate. The data cannot tell `T_c = 21`
from `T_c = 24`. An optimiser will return *one* of these and a standard error, and both are
meaningless. **Report the range, not the point.** This is exactly what happens on the real
data, where only a few hundred hours are above 22 °C.

## 9. `brentq`: invert a curve

A toy supply curve `price = 20 + 3 · x`. Which `x` gives a price of 200? `brentq` finds the
root of `f(x) = price(x) − 200` inside a bracket `[lo, hi]` where `f` changes sign.

In [23]:
def price_of(x_):
    return 20 + 3 * x_

def f(x_):
    return price_of(x_) - 200

lo, hi = 0.0, 100.0
print("f(lo) =", f(lo), "  f(hi) =", f(hi), "  -> opposite signs, bracket is valid")
root = optimize.brentq(f, lo, hi)
print("root:", root, "  check price_of(root) =", price_of(root))

f(lo) = -180.0   f(hi) = 120.0   -> opposite signs, bracket is valid
root: 60.0   check price_of(root) = 200.0


In [24]:
try:
    optimize.brentq(f, 0.0, 10.0)          # f(0) = -180, f(10) = -150: same sign
except ValueError as err:
    print("ValueError:", err)

ValueError: f(a) and f(b) must have different signs


## 10. Scaling: very large or very small numbers break optimisers

**Pitfall:** the model `y = a + b · exp(c · x)` with `x` in the thousands. Starting at
`c = 1`, `exp(1 · 3000)` overflows to `inf` on the very first evaluation. `curve_fit` then
hands back the starting values **unchanged, without raising**.

In [25]:
xs = np.array([1000.0, 2000.0, 3000.0, 4000.0, 5000.0])
ys = 1 + 2 * np.exp(0.0005 * xs)
print("ys:", ys)

def exp_model(x_, a, b, c):
    return a + b * np.exp(c * x_)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    popt_raw, _ = optimize.curve_fit(exp_model, xs, ys, p0=[1, 1, 1], maxfev=2000)
print("popt:", popt_raw, " <- identical to p0: the fit silently failed")

ys: [ 4.2974  6.4366  9.9634 15.7781 25.365 ]
popt: [1. 1. 1.]  <- identical to p0: the fit silently failed


Rescale the input so the exponent is a small number: `z = x / 1000`. Same model, same data,
now it works from the same generic start.

In [26]:
zs = xs / 1000
popt_z, _ = optimize.curve_fit(exp_model, zs, ys, p0=[1, 1, 1])
print("popt on scaled input:", popt_z)
print("c in original units :", popt_z[2] / 1000, "(true 0.0005)")

popt on scaled input: [1.  2.  0.5]
c in original units : 0.0004999999999999997 (true 0.0005)


The same problem appears when two parameters differ by a factor of a million. `minimize`
takes steps of similar size in every direction, so the step that is right for `a ≈ 1e6` is
absurd for `b ≈ 1`. Compare the number of function evaluations before and after rescaling.

In [27]:
def bowl(theta):
    a, b = theta
    return ((a - 1e6) / 1e6) ** 2 * 1e6 + (b - 1.0) ** 2

r1 = optimize.minimize(bowl, x0=[0.0, 0.0], method="Nelder-Mead")
print("unscaled: x =", r1.x, " nfev =", r1.nfev, " success =", r1.success)

def bowl_scaled(theta):            # a is measured in millions
    a_m, b = theta
    return bowl([a_m * 1e6, b])

r2 = optimize.minimize(bowl_scaled, x0=[0.0, 0.0], method="Nelder-Mead")
print("scaled  : x =", r2.x, " (a = ", r2.x[0] * 1e6, ")  nfev =", r2.nfev, " success =", r2.success)

unscaled: x = [1000000.       1.]  nfev = 280  success = True
scaled  : x = [1.     0.9999]  (a =  1000000.0096373047 )  nfev = 166  success = True


Both converge here, but the unscaled version needs many more evaluations; with more parameters
or a stricter method it fails outright.

Rule: before fitting, put every input and every parameter roughly in the range −10 … 10.
`least_squares` also accepts `x_scale=[...]` to tell it the natural size of each parameter.

## 11. Maximum likelihood with `minimize`

When the objective is not a sum of squares, write the negative log-likelihood and minimise it.
Toy: six numbers assumed normal, unknown mean `μ` and standard deviation `σ`.

In [28]:
data = np.array([4.0, 5.0, 7.0, 6.0, 5.0, 3.0])

def nll(theta):
    mu, log_sigma = theta            # optimise log(sigma) so sigma can never go negative
    sigma = np.exp(log_sigma)
    z = (data - mu) / sigma
    return len(data) * np.log(sigma) + 0.5 * (z ** 2).sum()

print("nll at mu=0, sigma=1 :", round(nll([0.0, np.log(1.0)]), 3))
print("nll at mu=5, sigma=1 :", round(nll([5.0, np.log(1.0)]), 3))
print("nll at mu=5, sigma=1.3:", round(nll([5.0, np.log(1.3)]), 3))

nll at mu=0, sigma=1 : 80.0
nll at mu=5, sigma=1 : 5.0
nll at mu=5, sigma=1.3: 4.533


In [29]:
r = optimize.minimize(nll, x0=[0.0, 0.0], method="L-BFGS-B")
print("success:", r.success, "|", r.message)
print("mu   :", round(r.x[0], 4), "   (sample mean:", data.mean(), ")")
print("sigma:", round(np.exp(r.x[1]), 4), "   (np.std, ddof=0:", round(np.std(data), 4), ")")

success: True | CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
mu   : 5.0    (sample mean: 5.0 )
sigma: 1.291    (np.std, ddof=0: 1.291 )


The maximum-likelihood estimates equal the sample mean and the population standard deviation
(divide by n, not n−1), as theory says.

**Pitfall:** `minimize` returns a result object even when it fails. Print `success` and
`message` every time; `r.x` is just where it stopped.

In [30]:
r_fail = optimize.minimize(nll, x0=[1000.0, 20.0], method="L-BFGS-B", options={"maxiter": 2})
print("success:", r_fail.success, "|", r_fail.message)
print("x:", r_fail.x, " nll:", round(r_fail.fun, 2), " (good nll was", round(r.fun, 2), ")")

success: False | STOP: TOTAL NO. of ITERATIONS REACHED LIMIT
x: [999.9991   6.9815]  nll: 44.45  (good nll was 4.53 )


## 12. The same fit on real data

Hourly consumption against temperature. Heating below about 15 °C, cooling above about 22 °C:

`consumption = a + b · max(T_h − T, 0) + c · max(T − T_c, 0)`

First remove the hour-of-day and weekend level (their group means), so temperature is the
only thing left to explain.

In [31]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
df["hour"] = df["time"].dt.hour
df["is_weekend"] = (df["time"].dt.dayofweek >= 5).astype(int)
group_mean = df.groupby(["hour", "is_weekend"])["consumption_mwh"].transform("mean")
df["resid_hw"] = df["consumption_mwh"] - group_mean
df[["time", "consumption_mwh", "temp_c", "resid_hw"]].head()

,time,consumption_mwh,temp_c,resid_hw
0,2022-01-01 00:00:00+00:00,26858.4,0.11,2882.930476
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,3288.860000
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,3796.200952
3,2022-01-01 03:00:00+00:00,25381.3,-0.75,3183.953810
4,2022-01-01 04:00:00+00:00,25223.0,-0.09,2777.821905


In [32]:
print("hours below 15C:", (df["temp_c"] < 15).sum())
print("hours above 22C:", (df["temp_c"] > 22).sum(), "  of", len(df))

hours below 15C: 12864
hours above 22C: 401   of 17520


Only a few hundred hours are above 22 °C: expect the cooling breakpoint to be poorly identified
(section 8). Fit with `curve_fit`.

In [33]:
def hdd_cdd(T, a, b, c, Th, Tc):
    return a + b * np.maximum(Th - T, 0) + c * np.maximum(T - Tc, 0)

T = df["temp_c"].values
yr = df["resid_hw"].values
p0 = [0, 300, 100, 14, 21]
popt, pcov = optimize.curve_fit(hdd_cdd, T, yr, p0=p0)
pd.DataFrame({"estimate": popt, "std_err": np.sqrt(np.diag(pcov)), "true": [np.nan, 380, 120, 15, 22]},
             index=["a", "b_heat", "c_cool", "T_heat", "T_cool"]).round(2)

,estimate,std_err,true
a,-2283.12,127.30,NaN
b_heat,360.87,1.65,380.0
c_cool,88.38,5.99,120.0
T_heat,15.24,0.35,15.0
T_cool,15.18,1.41,22.0


Heating side: slope and breakpoint close to the truth. Cooling side: `T_cool` collapsed onto
`T_heat`, a different model that fits almost as well (compare section 3: converged, but wrong).

Profile the SSE over `T_cool` with everything else re-fitted, as in section 8.

In [34]:
rows = []
for Tc in [17.0, 19.0, 21.0, 23.0, 25.0]:
    def resid_fixed(theta):
        a, b, c, Th = theta
        return hdd_cdd(T, a, b, c, Th, Tc) - yr
    r = optimize.least_squares(resid_fixed, x0=[0, 300, 100, 14])
    rows.append([Tc, round(r.x[2], 1), 2 * r.cost])
prof = pd.DataFrame(rows, columns=["T_cool fixed", "c_cool fitted", "SSE"])
prof["SSE relative to min"] = (prof["SSE"] / prof["SSE"].min() - 1).round(5)
prof

,T_cool fixed,c_cool fitted,SSE,SSE relative to min
0,17.0,99.1,1.537034e+10,0.00000
1,19.0,135.4,1.540701e+10,0.00239
2,21.0,202.4,1.546147e+10,0.00593
3,23.0,284.0,1.552542e+10,0.01009
4,25.0,375.1,1.556112e+10,0.01241


Across eight degrees of `T_cool` the SSE moves by about 1 %, and the minimum is at the lowest
value tried: the fit keeps pulling `T_cool` down towards `T_heat`. Compare the toy in section 7,
where two degrees of `T_h` changed the SSE by a factor of ten. The cooling breakpoint is barely
identified here. Report "somewhere between 17 and 25", not "21.3 ± 1.4".

Residual autocorrelation, for the standard-error inflation of section 5:

In [35]:
e_real = hdd_cdd(T, *popt) - yr
rho_real = np.corrcoef(e_real[1:], e_real[:-1])[0, 1]
n_eff_real = len(e_real) * (1 - rho_real) / (1 + rho_real)
print("rho:", round(rho_real, 3), " n:", len(e_real), " n_eff:", round(n_eff_real), " inflate se by:", round(np.sqrt(len(e_real) / n_eff_real), 1))

rho: 0.859  n: 17520  n_eff: 1325  inflate se by: 3.6


Finally the honest check: fit on 2022 only, evaluate on 2023.

In [36]:
is_2022 = (df["time"].dt.year == 2022).values
popt22, _ = optimize.curve_fit(hdd_cdd, T[is_2022], yr[is_2022], p0=p0)
pred22 = hdd_cdd(T[is_2022], *popt22)
pred23 = hdd_cdd(T[~is_2022], *popt22)
rmse22 = np.sqrt(np.mean((pred22 - yr[is_2022]) ** 2))
rmse23 = np.sqrt(np.mean((pred23 - yr[~is_2022]) ** 2))
bias23 = np.mean(yr[~is_2022] - pred23)
print("fit on 2022:", popt22.round(1))
print("RMSE 2022 (in-sample)    :", round(rmse22, 1))
print("RMSE 2023 (out-of-sample):", round(rmse23, 1))
print("mean error 2023          :", round(bias23, 1))

fit on 2022: [-2170.4   369.     78.8    14.9    14.9]
RMSE 2022 (in-sample)    : 930.1
RMSE 2023 (out-of-sample): 949.1
mean error 2023          : -139.1


The 2023 RMSE is only slightly higher, but the mean error is −139 MWh: the model over-predicts
on average. Where? Look at the mean error by month.

**Interview check:** "The RMSE barely changed, so the model is fine?" Not if the errors have a
pattern. A sign that flips with the season points at a misspecified term; a flat bias with
fatter tails means 2023 was simply noisier.

In [37]:
err23 = pd.Series(yr[~is_2022] - pred23, index=df.loc[~is_2022, "time"])
err23.groupby(err23.index.month).mean().round(0).rename("mean error by month, 2023")

time
1      19.0
2      97.0
3     110.0
4      15.0
5    -171.0
6    -442.0
7    -448.0
8    -288.0
9    -197.0
10      7.0
11   -106.0
12   -247.0
Name: mean error by month, 2023, dtype: float64

The error is strongly negative in June to September: the 2022 fit, with its cooling breakpoint
collapsed onto the heating one, over-predicts warm hours. The RMSE hid it; the monthly bias
table shows it. That is what this check is for.

## Quick reference

| Task | Tool | Watch out |
|---|---|---|
| Fit `y = f(x, θ)` | `curve_fit(f, x, y, p0, bounds)` | always pass `p0`; `pcov` assumes independent errors |
| Same, more control | `least_squares(res_fn, x0, bounds, loss, f_scale)` | check `status`, `message`, parameters on a bound |
| Any scalar objective | `minimize(fun, x0, method)` | check `success`; `x` is returned even on failure |
| Linear in some parameters | `lstsq` inside, `minimize_scalar` / `minimize` outside | far fewer non-linear parameters |
| Is a parameter identified? | profile the SSE over a grid | flat profile → report a range |
| Outliers | `loss="soft_l1"`, `f_scale` in residual units | inspect the down-weighted points |
| Root / inversion | `brentq(f, lo, hi)` | `f(lo)` and `f(hi)` must have opposite signs |
| Standard errors | `σ² (JᵀJ)⁻¹`, inflate by `sqrt(n / n_eff)` | or block bootstrap |
| Scaling | inputs and parameters roughly in [−10, 10] | `exp(large)` overflows silently |
| Validation | fit one year, test the next, bias by month | trends extrapolate |